<a href="https://colab.research.google.com/github/Joao-Vitor2/cdia-projetoAplicado3/blob/main/Colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import polars as pl
import json
import unicodedata
import os
from datetime import datetime
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# from config import DADOS_DIR, DUMP_DIR ## para deploy oficial

In [ ]:
pl.Config.set_tbl_cols(-1) # mostrar todas as COLUNAS do DataFrame

polars.config.Config

In [ ]:
def csvs_para_parquet():
    """Converte todos os CSVs da pasta atual para Parquet (mesmo nome, extensão .parquet)."""
    for caminho_csv in Path('.').glob('*.csv'):
        caminho_parquet = caminho_csv.with_suffix('.parquet')
        if caminho_parquet.exists():
            print(f'{caminho_parquet.name} já existe, pulando.')
            continue

        df = pl.read_csv(caminho_csv, separator=';', encoding='latin1', low_memory=True, rechunk=False)
        df.write_parquet(caminho_parquet)

        tamanho_csv_mb = caminho_csv.stat().st_size / 1e6
        tamanho_parquet_mb = caminho_parquet.stat().st_size / 1e6
        print(f'{caminho_csv.name} ({tamanho_csv_mb:.1f} MB) -> {caminho_parquet.name} ({tamanho_parquet_mb:.1f} MB)')

csvs_para_parquet()

ITENS_PROVA_2025.parquet já existe, pulando.
RESULTADOS_2025.parquet já existe, pulando.
PARTICIPANTES_2025.parquet já existe, pulando.


In [ ]:
def dataframe_microdados(ano:int, tipo_dado:str, teste:bool = False) -> pl.DataFrame:
    # CAMINHO_PARQUET = f'{DADOS_DIR}/dados_{ano}/{tipo_dado}_{ano}.parquet'
    CAMINHO_PARQUET = f'{tipo_dado}_{ano}.parquet'

    if teste:
        df = pl.read_parquet(CAMINHO_PARQUET)
    else:
        df = pl.read_parquet(CAMINHO_PARQUET)

    return df

df_resultados = dataframe_microdados(2023, 'RESULTADOS')

NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ENSINO,IN_TREINEIRO,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,CO_UF_PROVA,SG_UF_PROVA,Q001,Q002,Q003,Q004,Q005,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023
i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
210066506229,2025,6,"""F""",1,2,2,1,4,null,0,2932200,"""Ubaitaba""",29,"""BA""","""C""","""F""","""A""","""A""",1,"""B""","""B""","""A""","""B""","""B""","""A""","""A""","""B""","""A""","""A""","""A""","""A""","""A""","""A""","""A""","""A""","""B""","""A"""


In [ ]:
def dataframes_inse():
    df_inse_escola = pl.DataFrame(pl.read_excel('inse_escolas.xlsx)')
    df_inse_municipio = pl.DataFrame(pl.read_excel('inse_municipios.xlsx'))

    return df_inse_escola, df_inse_municipio


df_inse_escola, df_inse_municipio = carregar_inse()

In [ ]:
def resultados_nivel(df_resultados, df_df_inse_escola, df_inse_municipio):
    #renomear colunas escolas
    df_inse_escola = df_inse_escola.rename({
        'INSE_CLASSIFICACAO': 'NIVEL_DA_ESCOLA',
        'INSE_VALOR_ABSOLUTO': 'VALOR_DA_ESCOLA',
    })

    df_inse_municipio = df_inse_municipio.rename({
        'INSE_CLASSIFICACAO': 'NIVEL_DO_MUNICIPIO',
        'INSE_VALOR_ABSOLUTO': 'VALOR_DO_MUNICIPIO',
    })

    # união dos resultados com os dados das escolas do INSE
    df_nivel = df_resultados.join(
        df_inse_escola,
        left_on='CO_ESCOLA',
        right_on='CO_ENTIDADE',
        how='left',
    )

    # união dos resultados com os dados dos municípios do INSE
    df_nivel = df_nivel.join(
        df_inse_municipio,
        left_on='CO_MUNICIPIO_ESC',
        right_on='CO_MUNICIPIO',
        how='left',
    )

    # depois desses dois joins, resultados tem DUAS colunas de nível:
    #   NIVEL_DA_ESCOLA    -> só preenchida pra quem tem escola conhecida (~36% das linhas)
    #   NIVEL_DO_MUNICIPIO -> preenchida pra quase todo mundo (~100% das linhas)

    # criar uma coluna de nível final, priorizando os dados de escolas (mais específico) do que municipios (mais geral)
    niveis_da_escola = df_nivel['NIVEL_DA_ESCOLA'].to_list()
    niveis_do_municipio = df_nivel['NIVEL_DO_MUNICIPIO'].to_list()
    valores_da_escola = df_nivel['VALOR_DA_ESCOLA'].to_list()
    valores_do_municipio = df_nivel['VALOR_DO_MUNICIPIO'].to_list()

    nivel_final = []
    valor_final = []

    for i in range(len(niveis_da_escola)):
        if niveis_da_escola[i] is not None:
            # essa pessoa tem escola conhecida, usa o nível da escola
            nivel_final.append(niveis_da_escola[i])
            valor_final.append(valores_da_escola[i])
        else:
            # não tem escola conhecida, usa o nível do município como aproximação
            nivel_final.append(niveis_do_municipio[i])
            valor_final.append(valores_do_municipio[i])

    # colocar essas duas listas de volta como colunas novas na tabela
    df_nivel = df_nivel.with_columns(
        pl.Series(name='NIVEL_ESTIMADO', values=nivel_final),
        pl.Series(name='VALOR_ESTIMADO', values=valor_final),
    )

    # verificar resultado
    print(df_nivel.select(['CO_ESCOLA', 'NIVEL_DA_ESCOLA', 'NIVEL_DO_MUNICIPIO', 'NIVEL_ESTIMADO']).head(10))
    print(f'Total de candidatos: {df_nivel.height}')
    print(f'Candidatos sem nenhum nível encontrado: {df_nivel.filter(pl.col('NIVEL_ESTIMADO').is_null()).height}')

    return df_nivel

df_nivel = resultados_nivel(df_resultados, df_df_inse_escola, df_inse_municipio)

In [ ]:
def calcular_media_por_nivel(df_resultados):
    # Passo 1: separar só quem tem um nível estimado (descarta os nulos)
    df_resultados_com_nivel = df_resultados.filter(pl.col('NIVEL_ESTIMADO').is_not_null())

    # Passo 2: agrupar por nível e calcular a média de cada nota,
    # uma coluna de cada vez (sem comprehension, pra ficar fácil de ler)
    media_por_nivel = df_resultados_com_nivel.group_by('NIVEL_ESTIMADO').agg(
        pl.col('NU_NOTA_CN').mean().round(1).alias('MEDIA_CN'),
        pl.col('NU_NOTA_CH').mean().round(1).alias('MEDIA_CH'),
        pl.col('NU_NOTA_LC').mean().round(1).alias('MEDIA_LC'),
        pl.col('NU_NOTA_MT').mean().round(1).alias('MEDIA_MT'),
        pl.col('NU_NOTA_REDACAO').mean().round(1).alias('MEDIA_REDACAO'),
        pl.len().alias('QUANTIDADE_DE_CANDIDATOS'),
    )

    # Passo 3: ordenar do nível mais baixo pro mais alto
    media_por_nivel = media_por_nivel.sort('NIVEL_ESTIMADO')

    print(media_por_nivel)
    return media_por_nivel


media_por_nivel = calcular_media_por_nivel(df_nivel)

In [ ]:
def validar_correlacao_por_escola(df_resultados):
    # Passo 1: pegar só as linhas que têm escola e Inse da escola conhecidos
    dados_com_escola = df_resultados.filter(
        pl.col('CO_ESCOLA').is_not_null() & pl.col('VALOR_DA_ESCOLA').is_not_null()
    )

    # Passo 2: para cada escola, pegar o Inse dela e a nota média de matemática
    por_escola = dados_com_escola.group_by('CO_ESCOLA').agg(
        pl.col('VALOR_DA_ESCOLA').first().alias('INSE_MEDIO'),
        pl.col('NU_NOTA_MT').mean().alias('NOTA_MEDIA_MT'),
        pl.len().alias('QUANTIDADE_DE_CANDIDATOS'),
    )

    # Passo 3: tirar escolas com poucos candidatos (a média fica instável)
    por_escola = por_escola.filter(pl.col('QUANTIDADE_DE_CANDIDATOS') >= 5)

    # Passo 4: calcular a correlação entre Inse e nota
    correlacao = por_escola.select(pl.corr('INSE_MEDIO', 'NOTA_MEDIA_MT')).item()
    print(f'Correlação Inse x nota (por escola, n={por_escola.height} escolas): {correlacao:.3f}')

    # Passo 5: fazer o gráfico de dispersão
    plt.figure(figsize=(7, 5))
    plt.scatter(por_escola['INSE_MEDIO'], por_escola['NOTA_MEDIA_MT'], alpha=0.3, s=10)
    plt.xlabel('Inse médio da escola')
    plt.ylabel('Nota média de Matemática da escola')
    plt.title(f'Inse x Nota (r = {correlacao:.2f})')
    plt.tight_layout()
    plt.show()

    return por_escola, correlacao


por_escola, correlacao = validar_correlacao_por_escola(df_nivel)

In [ ]:
def estimar_posicao_candidato(nota_usuario, coluna_nota, co_municipio, co_escola=None):
    # Passo 1: tentar achar o nível pela ESCOLA, se o usuário souber o código dela
    nivel = None
    if co_escola is not None:
        linha_escola = inse_escola.filter(pl.col('CO_ENTIDADE') == co_escola)
        if linha_escola.height > 0:
            nivel = linha_escola['NIVEL_DA_ESCOLA'][0]

    # Passo 2: se não achou pela escola, usar o nível do MUNICÍPIO
    if nivel is None:
        linha_municipio = inse_municipio.filter(pl.col('CO_MUNICIPIO') == co_municipio)
        if linha_municipio.height == 0:
            raise ValueError(f'Município {co_municipio} não encontrado na planilha de Inse.')
        nivel = linha_municipio['NIVEL_DO_MUNICIPIO'][0]

    # Passo 3: pegar todo mundo em RESULTADOS que tem esse mesmo nível
    grupo = resultados.filter(
        (pl.col('NIVEL_ESTIMADO') == nivel) & pl.col(coluna_nota).is_not_null()
    )

    # Passo 4: calcular a média do grupo e o percentil do usuário dentro dele
    media_grupo = grupo[coluna_nota].mean()
    percentil = (grupo[coluna_nota] < nota_usuario).mean() * 100

    # Passo 5: devolver tudo organizado num dicionário
    return {
        'nivel_inse': nivel,
        'tamanho_grupo': grupo.height,
        'media_grupo': round(media_grupo, 1),
        'nota_usuario': nota_usuario,
        'diferenca_absoluta': round(nota_usuario - media_grupo, 1),
        'percentil_usuario': round(percentil, 1),
    }


# exemplo de uso:
# resultado = estimar_posicao_candidato(
#     nota_usuario=680.5, coluna_nota='NU_NOTA_MT',
#     co_municipio=3550308, co_escola=None,
# )
# print(resultado)